In [1]:
import sys
import joblib
import torch
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
cwd = Path.cwd()
project_root = cwd.parent
if project_root not in sys.path:
    sys.path.insert(0, str(project_root))
    print("Done!")

Done!


In [3]:
from credit_risk.dataset import load_splits
from credit_risk.config import PROCESSED_DATA_DIR

2026-06-29 18:07:22.140 | INFO     | credit_risk.config:<module>:11 - PROJ_ROOT path is: /Users/ak007/SML/Credit-Risk-Default-Prediction-System


In [4]:
train, val, test, _ = load_splits(PROCESSED_DATA_DIR / "after_eda")

2026-06-02 08:46:35.585 | INFO     | credit_risk.dataset:load_splits:260 - Checking if the files exists...
2026-06-02 08:46:35.593 | INFO     | credit_risk.dataset:load_splits:262 - Loading the Cached files...
2026-06-02 08:46:35.851 | INFO     | credit_risk.dataset:load_splits:270 - Loaded sucessfully all the splits and the metadata, Train_df shape: (466071, 110), val_df shape: (420204, 110), test_df shape: (431712, 110)


In [5]:
check = train.isna().all()

In [6]:
check[check]

annual_inc_joint                       True
dti_joint                              True
verification_status_joint              True
open_acc_6m                            True
open_act_il                            True
open_il_12m                            True
open_il_24m                            True
mths_since_rcnt_il                     True
total_bal_il                           True
il_util                                True
open_rv_12m                            True
open_rv_24m                            True
max_bal_bc                             True
all_util                               True
inq_fi                                 True
total_cu_tl                            True
inq_last_12m                           True
revol_bal_joint                        True
sec_app_fico_range_low                 True
sec_app_fico_range_high                True
sec_app_earliest_cr_line               True
sec_app_inq_last_6mths                 True
sec_app_mort_acc                

In [15]:
for name, df in [("train", train), ("val", val), ("test", test)]:
    fully_null = df.columns[df.isna().all()].tolist()
    print(f"{name}: {len(fully_null)} fully-null columns")
    print(f"  {fully_null}")

train: 30 fully-null columns
  ['annual_inc_joint', 'dti_joint', 'verification_status_joint', 'open_acc_6m', 'open_act_il', 'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il', 'total_bal_il', 'il_util', 'open_rv_12m', 'open_rv_24m', 'max_bal_bc', 'all_util', 'inq_fi', 'total_cu_tl', 'inq_last_12m', 'revol_bal_joint', 'sec_app_fico_range_low', 'sec_app_fico_range_high', 'sec_app_earliest_cr_line', 'sec_app_inq_last_6mths', 'sec_app_mort_acc', 'sec_app_open_acc', 'sec_app_revol_util', 'sec_app_open_act_il', 'sec_app_num_rev_accts', 'sec_app_chargeoff_within_12_mths', 'sec_app_collections_12_mths_ex_med', 'sec_app_mths_since_last_major_derog']
val: 13 fully-null columns
  ['revol_bal_joint', 'sec_app_fico_range_low', 'sec_app_fico_range_high', 'sec_app_earliest_cr_line', 'sec_app_inq_last_6mths', 'sec_app_mort_acc', 'sec_app_open_acc', 'sec_app_revol_util', 'sec_app_open_act_il', 'sec_app_num_rev_accts', 'sec_app_chargeoff_within_12_mths', 'sec_app_collections_12_mths_ex_med', 'sec_app_mt

In [16]:
joint_loans = train[train['application_type'] == 'Joint App']
print(f"Joint apps in train: {len(joint_loans)}")
print(joint_loans[['revol_bal_joint', 'sec_app_fico_range_high', 'annual_inc_joint']].head())

Joint apps in train: 0
Empty DataFrame
Columns: [revol_bal_joint, sec_app_fico_range_high, annual_inc_joint]
Index: []


In [17]:
for name, df in [("train", train), ("val", val), ("test", test)]:
    print(f"\n{name}:")
    print(df['application_type'].value_counts(dropna=False))


train:
application_type
Individual    466071
Name: count, dtype: int64

val:
application_type
Individual    419694
Joint App        510
Name: count, dtype: int64

test:
application_type
Individual    423032
Joint App       8680
Name: count, dtype: int64


In [9]:
train.select_dtypes(include='str').columns

Index(['term', 'grade', 'sub_grade', 'emp_title', 'emp_length',
       'home_ownership', 'verification_status', 'desc', 'purpose', 'title',
       'zip_code', 'addr_state', 'earliest_cr_line', 'initial_list_status',
       'application_type', 'verification_status_joint',
       'sec_app_earliest_cr_line', 'disbursement_method'],
      dtype='str')

In [32]:
set(train['emp_length'].dropna().unique()).union(set(val['emp_length'].dropna().unique())).union(set(test['emp_length'].dropna().unique()))

{'1 year',
 '10+ years',
 '2 years',
 '3 years',
 '4 years',
 '5 years',
 '6 years',
 '7 years',
 '8 years',
 '9 years',
 '< 1 year'}

In [11]:
train['term'].value_counts()

term
36 months    337996
60 months    128075
Name: count, dtype: int64

In [3]:
from credit_risk.features import FEATURES_DIR, load_features

2026-06-29 22:03:01.075 | INFO     | credit_risk.config:<module>:11 - PROJ_ROOT path is: /Users/ak007/SML/Credit-Risk-Default-Prediction-System


In [4]:
feature_splits = load_features()

2026-06-29 22:03:01.774 | INFO     | credit_risk.features:load_features:263 - Loading the processed features...
2026-06-29 22:03:02.273 | INFO     | credit_risk.features:load_features:270 - Loaded successfully!


In [5]:
X_train = feature_splits['train'][0].to_numpy()
y_train = feature_splits['train'][1].to_numpy().ravel()
X_val = feature_splits['val'][0].to_numpy()
y_val = feature_splits['val'][1].to_numpy().ravel()
X_test = feature_splits['test'][0].to_numpy()
y_test = feature_splits['test'][1].to_numpy().ravel()

In [6]:
# loading paths
lr_path = project_root / 'models' / 'baseline_lr'
xgb_path = project_root / 'models' / 'tuned_xgb'
mlp_path = project_root / 'models' / 'mlp'

In [7]:
# loading lr and tuned_xgb
lr_model = joblib.load(filename=lr_path / 'model.pkl')
tuned_xgb = joblib.load(filename=xgb_path / 'model.pkl')

In [17]:
xgb_proba = tuned_xgb[1].predict_proba(X_val)[:,1]

In [9]:
mlp_proba = np.load(mlp_path / 'val_proba.npy')

In [10]:
from sklearn.linear_model import LogisticRegression

In [11]:
lr_model_new = LogisticRegression(max_iter=1000)
lr_model_new.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [18]:
lr_proba = lr_model_new.predict_proba(X_val)[:,1]

In [13]:
lr_p  = lr_proba[:, 1]
xgb_p = xgb_proba[:, 1]
mlp_p = mlp_proba

In [14]:
corr_lr_xgb  = np.corrcoef(lr_p, xgb_p)[0, 1]
corr_lr_mlp  = np.corrcoef(lr_p, mlp_p)[0, 1]
corr_xgb_mlp = np.corrcoef(xgb_p, mlp_p)[0, 1]

In [15]:
corr_lr_mlp, corr_lr_xgb, corr_xgb_mlp

(np.float64(0.8708495027253107),
 np.float64(0.9244796432513082),
 np.float64(0.8946618420404652))

In [19]:
lr_train_proba = lr_model_new.predict_proba(X_train)[:,1]
lr_test_proba = lr_model_new.predict_proba(X_test)[:,1]
xgb_train_proba = tuned_xgb[1].predict_proba(X_train)[:,1]
xgb_test_proba = tuned_xgb[1].predict_proba(X_test)[:,1]
mlp_train_proba = np.load(mlp_path / 'train_proba.npy')
mlp_test_proba = np.load(mlp_path / 'test_proba.npy')

In [21]:
from credit_risk.evaluation import evaluate_model

In [22]:
ens_train_proba = (lr_train_proba + xgb_train_proba + mlp_train_proba) / 3
ens_val_proba = (lr_proba + xgb_proba + mlp_proba) / 3
ens_test_proba = (lr_test_proba + xgb_test_proba + mlp_test_proba) / 3

# Evaluate on both with that fixed threshold
ens_results = evaluate_model(
    y_train, ens_train_proba,
    y_val, ens_val_proba, y_test, ens_test_proba,
    fn_cost=10000, fp_cost=2000,
)

In [23]:
ens_results

{'threshold': np.float64(0.16),
 'train': {'ROC-AUC': 0.7323397848262886,
  'PR-AUC': 0.3634314540304757,
  'brier_score': 0.1247266067685922,
  'precision': 0.27318974500127796,
  'recall': 0.7162961531026484,
  'confusion_matrix': array([[240576, 147871],
         [ 22014,  55581]])},
 'val': {'ROC-AUC': 0.7110102546143432,
  'PR-AUC': 0.3510984912063141,
  'brier_score': 0.1369648823369219,
  'precision': 0.2816037301757723,
  'recall': 0.713813615333774,
  'confusion_matrix': array([[202527, 140514],
         [ 22083,  55080]])},
 'test': {'ROC-AUC': 0.6972545661115351,
  'PR-AUC': 0.312510813896839,
  'brier_score': 0.13020872032325642,
  'precision': 0.2613729192250889,
  'recall': 0.6636720789032666,
  'confusion_matrix': array([[222381, 136533],
         [ 24484,  48314]])}}

In [11]:
import json
import time
from credit_risk.modeling.mlp import MLP

In [8]:
with open(mlp_path / 'architecture.json') as f:
    archi = json.load(f)

In [9]:
state_dict = torch.load(mlp_path / 'model_state.pt')
mlp_model = MLP(input_dim=archi['input_dim'], hidden_dim=archi['hidden_dim'], dropout=archi['dropout'])
mlp_model.load_state_dict(state_dict=state_dict)

<All keys matched successfully>

In [12]:
# mlp

single_batch = torch.from_numpy(X_test[:1000].astype(np.float32))

with torch.no_grad():
    # warmup
    for _ in range(3):
        _ = torch.sigmoid(mlp_model(single_batch))
        
    n_trials = 50
    times = []
    for _ in range(50):
        start = time.perf_counter()
        _ = torch.sigmoid(mlp_model(single_batch))
        end = time.perf_counter()
        
        times.append((end - start) * 1000)
        
print(f"Median: {np.median(times):.3f} ms")
print(f"per-row: {(np.median(times)/1000):.3f} ms")

Median: 0.632 ms
per-row: 0.001 ms


In [13]:
# MLP

single_tensor = torch.from_numpy(X_test[:1].astype(np.float32))

mlp_model.eval()
with torch.no_grad():
    # warm up
    for _ in range(10):
        _ = torch.sigmoid(mlp_model(single_tensor))
        
    n_trials = 100
    times = []
    for _ in range(n_trials):
        start = time.perf_counter()
        _ = torch.sigmoid(mlp_model(single_tensor))
        end = time.perf_counter()
        times.append((end - start) * 1000)
        
print(f"median: {np.median(times):.3f} ms")
print(f'P95: {np.percentile(times, 95):.3f} ms')

median: 0.035 ms
P95: 0.064 ms
